In [69]:
import pandas as pd

import sys
sys.path.append('..')

from config import RUTA_UNIDAD_ONE_DRIVE
from config import RUTA_LOCAL_ONE_DRIVE
from config import POSTGRES_UTEA

USER_DB = POSTGRES_UTEA['USER']
PASS_DB = POSTGRES_UTEA['PASSWORD']
HOST_DB = POSTGRES_UTEA['HOST']
PORT_DB = POSTGRES_UTEA['PORT']
NAME_DB = POSTGRES_UTEA['DATABASE']

In [70]:
# validad campos de grupos economicos
def validar_grupos_economicos(df):
    total_filas = len(df)
    
    # 1. Identificar nulos y vacíos
    # Convertimos strings con espacios a nulos reales para detectarlos
    df_temp = df.replace(r'^\s*$', pd.NA, regex=True)
    nulos_por_columna = df_temp.isnull().sum()
    
    # 2. Validar duplicados en CODIGO
    duplicados_codigo = df['CODIGO'].duplicated().sum()
    registros_duplicados = df[df['CODIGO'].duplicated(keep=False)].sort_values(by='CODIGO')
    
    # 3. Construcción del reporte
    print("==================================================")
    print("   REPORTE DE VALIDACIÓN: GRUPOS ECONÓMICOS")
    print("==================================================")
    print(f"Total de registros analizados: {total_filas}")
    print("\n[1] CONTEO DE VALORES NULOS O VACÍOS:")
    
    hay_nulos = False
    for col, cant in nulos_por_columna.items():
        if cant > 0:
            print(f"  - {col}: {cant} registros faltantes")
            hay_nulos = True
    
    if not hay_nulos:
        print("  - ¡Perfecto! No se encontraron campos vacíos.")

    print("\n[2] VALIDACIÓN DE UNICIDAD (Columna 'CODIGO'):")
    if duplicados_codigo > 0:
        print(f"  - ¡ATENCIÓN! Se encontraron {duplicados_codigo} códigos repetidos.")
        print("  - Los primeros códigos duplicados detectados son:")
        print(registros_duplicados[['CODIGO', 'NOMBRE']].head(6))
    else:
        print("  - ¡Éxito! Todos los códigos son únicos.")

    print("\n[3] RESUMEN DE GRUPOS:")
    if 'GRUPO ECONOMICO' in df.columns:
        cant_grupos = df['GRUPO ECONOMICO'].nunique()
        print(f"  - Cantidad de grupos económicos distintos: {cant_grupos}")
    
    print("==================================================")
    
# validar campos de GRUPOS_TRABAJO
def validar_grupos_trabajo(df):
    total_filas = len(df)
    
    # 1. Identificar nulos, vacíos y ceros
    # Convertimos vacíos a NA para conteo uniforme
    df_temp = df.replace(r'^\s*$', pd.NA, regex=True)
    
    # Conteos específicos
    nulos = df_temp.isnull().sum()
    # Solo contamos ceros en columnas numéricas para evitar errores con strings
    ceros = (df == 0).sum()
    
    # 2. Validar duplicados en CODIGO CAÑERO
    duplicados_canero = df['CODIGO CAÑERO'].duplicated().sum()
    registros_duplicados = df[df['CODIGO CAÑERO'].duplicated(keep=False)].sort_values(by='CODIGO CAÑERO')
    
    # 3. Conteo de Delegados
    total_delegados = df[df['DELEGADO'].notnull()].shape[0]

    # --- REPORTE ---
    print("==================================================")
    print("   REPORTE DE VALIDACIÓN: GRUPOS DE TRABAJO")
    print("==================================================")
    print(f"Total de registros: {total_filas}")
    
    print("\n[1] VALORES INVÁLIDOS (NULOS / VACÍOS / CEROS):")
    hay_errores = False
    
    # Iteramos sobre todas las columnas para mostrar el resumen
    todas_las_columnas = df.columns
    for col in todas_las_columnas:
        cant_nulos = nulos.get(col, 0)
        cant_ceros = ceros.get(col, 0)
        
        if cant_nulos > 0 or cant_ceros > 0:
            msg = f"  - {col}:"
            if cant_nulos > 0: msg += f" {cant_nulos} Nulos/Vacíos;"
            if cant_ceros > 0: msg += f" {cant_ceros} Valores en Cero;"
            print(msg)
            hay_errores = True
            
    if not hay_errores: 
        print("  - [OK] No se detectaron campos vacíos ni valores en cero.")

    print("\n[2] INTEGRIDAD DE CÓDIGOS (CODIGO CAÑERO):")
    # Validación extra: El código cañero no debería ser 0
    ceros_en_codigo = (df['CODIGO CAÑERO'] == 0).sum()
    
    if duplicados_canero > 0 or ceros_en_codigo > 0:
        if duplicados_canero > 0:
            print(f"  - [!] ALERTA: Hay {duplicados_canero} códigos de cañero repetidos.")
        if ceros_en_codigo > 0:
            print(f"  - [!] ALERTA: Hay {ceros_en_codigo} registros con CODIGO CAÑERO = 0.")
            
        print("  - Listado de conflictos (primeros 5):")
        # Filtramos duplicados o ceros para mostrar
        conflictos = df[(df['CODIGO CAÑERO'].duplicated(keep=False)) | (df['CODIGO CAÑERO'] == 0)]
        print(conflictos[['CODIGO CAÑERO', 'NOMBRE CAÑERO']].sort_values(by='CODIGO CAÑERO').head(5))
    else:
        print("  - [OK] Todos los códigos de cañero son únicos y válidos.")

    print("\n[3] RESUMEN DE ESTRUCTURA:")
    print(f"  - Registros marcados como DELEGADO: {total_delegados}")
    if 'GRUPO TRABAJO' in df.columns:
        print(f"  - Cantidad de Grupos de Trabajo distintos: {df['GRUPO TRABAJO'].nunique()}")
    
    print("==================================================")
    
    
# compara GRUPO_ECO con GRUPO_TRABAJO y retorna la diferencia
def obtener_diferencias_grupos(df_eco, df_trabajo):
    """
    Compara DF_GRUPOS_ECO y DF_GRUPOS_TRABAJO basándose en el código de cañero.
    Retorna: (solo_en_eco, solo_en_trabajo)
    """
    
    # 0. Asegurar que las columnas de cruce sean del mismo tipo (ej. int) para evitar errores de merge
    # Ignoramos nulos/ceros para la conversión si es necesario
    eco_copy = df_eco.copy()
    trabajo_copy = df_trabajo.copy()
    
    eco_copy['CODIGO'] = pd.to_numeric(eco_copy['CODIGO'], errors='coerce')
    trabajo_copy['CODIGO CAÑERO'] = pd.to_numeric(trabajo_copy['CODIGO CAÑERO'], errors='coerce')

    # 1. Realizar el merge con indicador
    df_comparativo = pd.merge(
        eco_copy, 
        trabajo_copy, 
        left_on='CODIGO', 
        right_on='CODIGO CAÑERO', 
        how='outer', 
        indicator=True
    )

    # 2. Separar registros que están en uno pero no en el otro
    # Filtramos por la columna técnica '_merge'
    df_solo_eco = df_comparativo[df_comparativo['_merge'] == 'left_only'].copy()
    df_solo_trabajo = df_comparativo[df_comparativo['_merge'] == 'right_only'].copy()

    # 3. Selección de columnas específicas para limpiar el resultado
    # Definimos las columnas originales de cada set
    cols_eco = ['DEUDOR', 'CODIGO', 'NOMBRE', 'GRUPO ECONOMICO', 'COD. INSTITUCION', 'ORDEN', 'CABEZA DEL GRUPO']
    cols_trabajo = ['CODIGO CAÑERO', 'NOMBRE CAÑERO', 'GRUPO TRABAJO', 'INS', 'DELEGADO']

    # Aplicamos el filtro de columnas (solo si existen en el resultado)
    df_solo_eco = df_solo_eco[cols_eco]
    df_solo_trabajo = df_solo_trabajo[cols_trabajo]

    # --- Reporte en consola ---
    print("--------------------------------------------------")
    print("SINCRONIZACIÓN DE GRUPOS FINALIZADA")
    print(f"Registros encontrados SOLO en ECO: {len(df_solo_eco)}")
    print(f"Registros encontrados SOLO en TRABAJO: {len(df_solo_trabajo)}")
    print("--------------------------------------------------")

    return df_solo_eco, df_solo_trabajo

In [77]:
# ruta de GRUPOS_ECONOMICOS
PATH_GRUPOS_ECO = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\ZAFRAS - ZAFRA 2026\UCAF\GRUPOS ECONOMICOS 2026.xlsx'
# cargar en dataframe los grupos eco
DF_GRUPOS_ECO = pd.read_excel(PATH_GRUPOS_ECO, sheet_name='grupos economicos')

# ruta de GRUPOS_TRABAJO
PATH_GRUPOS_TRABAJO = RUTA_UNIDAD_ONE_DRIVE + r'\Ingenio Azucarero Guabira S.A\UTEA - SEMANAL - AVANCE COSECHA\2026\DATA\GRUPO_TRABAJO.xlsx'
# cargar en dataframe los grupos de trabajo
DF_GRUPOS_TRABAJO = pd.read_excel(PATH_GRUPOS_TRABAJO, sheet_name='2026')

In [78]:
# VALIDAR CAMPOS DE GRUPSO ECONOMICOS
validar_grupos_economicos(DF_GRUPOS_ECO)

   REPORTE DE VALIDACIÓN: GRUPOS ECONÓMICOS
Total de registros analizados: 1330

[1] CONTEO DE VALORES NULOS O VACÍOS:
  - CABEZA DEL GRUPO: 535 registros faltantes

[2] VALIDACIÓN DE UNICIDAD (Columna 'CODIGO'):
  - ¡Éxito! Todos los códigos son únicos.

[3] RESUMEN DE GRUPOS:
  - Cantidad de grupos económicos distintos: 795


In [79]:
# VALIDAR CAMPOS DE GRUPSO TRABAJO
validar_grupos_trabajo(DF_GRUPOS_TRABAJO)

   REPORTE DE VALIDACIÓN: GRUPOS DE TRABAJO
Total de registros: 1331

[1] VALORES INVÁLIDOS (NULOS / VACÍOS / CEROS):
  - INS: 3 Valores en Cero;
  - DELEGADO: 785 Nulos/Vacíos;
  - Columna1: 3 Nulos/Vacíos;

[2] INTEGRIDAD DE CÓDIGOS (CODIGO CAÑERO):
  - [OK] Todos los códigos de cañero son únicos y válidos.

[3] RESUMEN DE ESTRUCTURA:
  - Registros marcados como DELEGADO: 546
  - Cantidad de Grupos de Trabajo distintos: 546


In [80]:
# COMPARAR AMBOS GRUPOS
solo_en_eco, solo_en_trabajo = obtener_diferencias_grupos(DF_GRUPOS_ECO, DF_GRUPOS_TRABAJO)

--------------------------------------------------
SINCRONIZACIÓN DE GRUPOS FINALIZADA
Registros encontrados SOLO en ECO: 2
Registros encontrados SOLO en TRABAJO: 3
--------------------------------------------------


In [81]:
# CAÑEROS QUE SOLO ESTAN EN GRUPO ECONOMICO Y NO EN TRABAJO
solo_en_eco

,DEUDOR,CODIGO,NOMBRE,GRUPO ECONOMICO,COD. INSTITUCION,ORDEN,CABEZA DEL GRUPO
598,800977.0,9850.0,OVANDO HUALLPA TIBURCIO,358.0,INST0063,1.0,CABEZA DE GRUPO
626,802230.0,42034.0,ACSAMA MAMANI TEODORA,382.0,INST0063,1.0,CABEZA DE GRUPO


In [82]:
# CAÑEROS QUE SOLO ESTAN EN GRUPO TRABAJO Y NO EN ECONOMICO
solo_en_trabajo

,CODIGO CAÑERO,NOMBRE CAÑERO,GRUPO TRABAJO,INS,DELEGADO
1330,11261.0,QUIROZ CUBA LIMBERG,14.0,0.0,NaN
1331,7843.0,LAMAS MELGAR NULBER HUGO,502.0,0.0,NaN
1332,13475.0,TORRICO LUIZAGA LIMBERG,504.0,0.0,NaN
